# Week 11

Two Hybrid recommender models using Parallel combination strategy 
one from SVD CF combined with content based from week 10  
the other SVD CF combined with content based model with gemma3 generated descriptions

In [1]:
%pip install ollama

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# load imports
import numpy as np
import pandas as pd
import pickle
import re
from pathlib import Path

from itertools import product

from scipy.sparse import csr_matrix, hstack, diags
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize, MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

from surprise import SVD
import ollama

# load user-item interactions
user_item_matrix = pd.read_csv('csv_files/user_item_matrix.csv', index_col=0)
# load train and test
train_df = pd.read_csv('csv_files/preprocessed_train_video_games.csv')
test_df = pd.read_csv('csv_files/preprocessed_test_video_games.csv')
metadata_df = pd.read_parquet('datasets/meta_video_games.parquet')


You can make the Option 2 more effcient by generating descriptions only
for the top-k (with a large enough k) items recommended by the collaborative filtering model, instead of all items in the dataset. Then, your hybrid approach
will re-rank the top-k items

### 1) Collaborative Filtering Base
1. Use your strongest CF setup from previous weeks (e.g., KNN or SVD). 
1b. use the saved SVD model that was pickled
2. Generate predicted scores for unrated (user, item) pairs.
3. Keep full candidate ranking or at least top-K candidates per user.

In [3]:
# load best svd model (user-item matrix with predictions)
u_i_matrix_svd = pd.read_csv("csv_files/user_item_matrix_svd.csv", index_col=0)
u_i_matrix_svd.info()
# check for null cells
u_i_matrix_svd.isnull().sum().sum()

# load unobserved predictions svd
unobserved_preds_svd = pd.read_csv("csv_files/unobserved_predictions_svd.csv", index_col=0)
# sort by rating in descending order
unobserved_preds_svd = unobserved_preds_svd.sort_values("svd_rating", ascending=False)
unobserved_preds_svd.head()

<class 'pandas.core.frame.DataFrame'>
Index: 1389 entries, AE25ZDXYBK3LHKCZ7XUODANPME4A to AHZYXDJ3HNLKS2E73VOSNIZZJT4Q
Columns: 932 entries, B00000JRSB to B0C5K4M7WJ
dtypes: float64(932)
memory usage: 9.9+ MB


,item_id,svd_rating
user_id,,
AELD7NXSVVFKTNFB3I673NGJVSWA,B00J4Y6L2Y,5.0
AF52HVGBBCP3JAS6B7B3I4AVIWRQ,B017W1771Y,5.0
AHMG3HUA4K6H475VQGJF2QAOTFTQ,B0BDWVBWC9,5.0
AF52HVGBBCP3JAS6B7B3I4AVIWRQ,B004QEV0MI,5.0
AGP7FLSJ7BHYFVZY3FRKVVDBH47Q,B007CSF3GO,5.0


#### DO a SUBSET for easier time matching common item between all the sets and lower compuational cost
how much to filter by out of the 927? 


### 2) Content-Based Base (Original Metadata)
1. Implement same CB representation as Week 10.
1b. rerun week 10 and save the content based model as pickle file
2. prompt an agent and ask how I can create an unobserved df frame from the predicitions in week 10

In [11]:
# Build content-based predictions for unobserved pairs (for hybrid in Week 11)

import numpy as np
import pandas as pd
import pickle

# 1) Load Week 10 trained artifacts
with open("models/best_content_based_model.pkl", "rb") as f:
    cb_artifacts = pickle.load(f)

user_id_to_row = cb_artifacts["user_id_to_row"]
item_id_to_row = cb_artifacts["item_id_to_row"]
user_matrix = cb_artifacts["user_matrix"]   # sparse
item_matrix = cb_artifacts["item_matrix"]   # sparse 


# 2) Load the same unobserved pair list used by SVD
# If your file has an index column, this still works safely.
unobs_pairs = unobserved_preds_svd.copy()

# this is because the csv headers are not read in properly
if "user_id" not in unobs_pairs.columns:
    unobs_pairs = unobs_pairs.reset_index().rename(columns={"index": "user_id"})

required = {"user_id", "item_id"}
missing = required.difference(unobs_pairs.columns)
if missing:
    raise ValueError(f"Missing required columns in unobserved file: {missing}")

unobs_pairs["user_id"] = unobs_pairs["user_id"].astype(str)
unobs_pairs["item_id"] = unobs_pairs["item_id"].astype(str)

# 3) Keep only pairs that exist in CB mappings
pairs = unobs_pairs[
    unobs_pairs["user_id"].isin(user_id_to_row) &
    unobs_pairs["item_id"].isin(item_id_to_row)
].copy()


# After the .isin() filter (NaNs are already gone), force int explicitly
pairs["u_row"] = pairs["user_id"].map(user_id_to_row).astype(int)
pairs["i_row"] = pairs["item_id"].map(item_id_to_row).astype(int)


print("user_matrix shape:", user_matrix.shape)
print("item_matrix shape:", item_matrix.shape)

print("\nu_row stats:")
print(pairs["u_row"].describe())
print("Negative u_rows:", (pairs["u_row"] < 0).sum())
print("u_row >= user_matrix.shape[0]:", (pairs["u_row"] >= user_matrix.shape[0]).sum())

print("\ni_row stats:")
print(pairs["i_row"].describe())
print("Negative i_rows:", (pairs["i_row"] < 0).sum())
print("i_row >= item_matrix.shape[0]:", (pairs["i_row"] >= item_matrix.shape[0]).sum())



user_matrix shape: (1389, 9606)
item_matrix shape: (927, 9606)

u_row stats:
count    1.261123e+06
mean     6.951193e+02
std      4.010045e+02
min      0.000000e+00
25%      3.480000e+02
50%      6.960000e+02
75%      1.043000e+03
max      1.388000e+03
Name: u_row, dtype: float64
Negative u_rows: 0
u_row >= user_matrix.shape[0]: 0

i_row stats:
count    1.261123e+06
mean     4.628827e+02
std      2.678035e+02
min      0.000000e+00
25%      2.310000e+02
50%      4.630000e+02
75%      6.950000e+02
max      9.260000e+02
Name: i_row, dtype: float64
Negative i_rows: 0
i_row >= item_matrix.shape[0]: 0


In [12]:
# Generate CB scores for all mappable unobserved pairs and save output (chunked to avoid sparse indexing overflow)
u_idx = pairs['u_row'].to_numpy(dtype=np.int64)
i_idx = pairs['i_row'].to_numpy(dtype=np.int64)

user_norms = np.sqrt(np.asarray(user_matrix.multiply(user_matrix).sum(axis=1)).ravel())
item_norms = np.sqrt(np.asarray(item_matrix.multiply(item_matrix).sum(axis=1)).ravel())

chunk_size = 50000
cos = np.empty(len(pairs), dtype=np.float32)

for start in range(0, len(pairs), chunk_size):
    end = min(start + chunk_size, len(pairs))
    uu = u_idx[start:end]
    ii = i_idx[start:end]

    U_chunk = user_matrix[uu]
    I_chunk = item_matrix[ii]

    dot_ui = np.asarray(U_chunk.multiply(I_chunk).sum(axis=1)).ravel()
    den = user_norms[uu] * item_norms[ii]

    c = np.divide(dot_ui, den, out=np.zeros_like(dot_ui, dtype=float), where=den > 0)
    cos[start:end] = np.clip(c, 0.0, 1.0)

    if (start // chunk_size + 1) % 10 == 0 or end == len(pairs):
        print(f'Processed {end:,}/{len(pairs):,} pairs')

cb_out = pairs[['user_id', 'item_id']].copy()
cb_out['cb_score_cosine'] = cos
cb_out['cb_rating'] = 1.0 + 4.0 * cos

out_path = 'csv_files/unobserved_predictions_cb.csv'
cb_out.to_csv(out_path, index=False)

print('Saved:', out_path)
print('Rows saved:', len(cb_out))
cb_out.head()

Processed 500,000/1,261,123 pairs
Processed 1,000,000/1,261,123 pairs
Processed 1,261,123/1,261,123 pairs
Saved: csv_files/unobserved_predictions_cb.csv
Rows saved: 1261123


,user_id,item_id,cb_score_cosine,cb_rating
0,AELD7NXSVVFKTNFB3I673NGJVSWA,B00J4Y6L2Y,0.270904,2.083617
1,AF52HVGBBCP3JAS6B7B3I4AVIWRQ,B017W1771Y,0.115265,1.461059
2,AHMG3HUA4K6H475VQGJF2QAOTFTQ,B0BDWVBWC9,0.128664,1.514656
3,AF52HVGBBCP3JAS6B7B3I4AVIWRQ,B004QEV0MI,0.328952,2.315807
4,AGP7FLSJ7BHYFVZY3FRKVVDBH47Q,B007CSF3GO,0.306324,2.225295


### 3) Content-Based Base Gemma3 (LLM Metadata)
1. For each item title, generate a short, structured description using Gemma via Ollama.
2. Create text features from generated descriptions (same vectorization pipeline used in CB A where possible).
3. Compute CB scores for unrated items.


In [ ]:
# Build SVD candidate pool using per-user top-k (candidate generation stage)
k_per_user = 10

unobs_svd = unobserved_preds_svd.copy()
if "user_id" not in unobs_svd.columns:
    unobs_svd = unobs_svd.reset_index()

required_cols = {"user_id", "item_id", "svd_rating"}
missing = required_cols.difference(unobs_svd.columns)
if missing:
    raise ValueError(f"Missing required columns in unobserved SVD predictions: {missing}")

unobs_svd["user_id"] = unobs_svd["user_id"].astype(str)
unobs_svd["item_id"] = unobs_svd["item_id"].astype(str)

topk_per_user_svd = (unobs_svd.sort_values(["user_id", "svd_rating"], ascending=[True, False]).groupby("user_id", as_index=False, group_keys=False).head(k_per_user).reset_index(drop=True))
# filter to unique items across all users, keeping the highest rating for each item
candidate_items_svd = (topk_per_user_svd.groupby("item_id", as_index=False)["svd_rating"].max().sort_values("svd_rating", ascending=False).reset_index(drop=True))

# remove duplicates
candidate_item_ids_svd = set(candidate_items_svd["item_id"].astype(str))

print(f"Users in SVD predictions: {topk_per_user_svd['user_id'].nunique()}")
print(f"Per-user top-k: {k_per_user}")
print(f"Rows kept after per-user top-k: {len(topk_per_user_svd):,}")
print(f"Unique candidate items: {len(candidate_item_ids_svd):,}")

candidate_items_svd.head(10)

Users in SVD predictions: 1389
Per-user top-k: 10
Rows kept after per-user top-k: 13,890
Unique candidate items: 546


,item_id,svd_rating
0,B00000JRSB,5.0
1,B00MB1I3FU,5.0
2,B00RU75I2G,5.0
3,B00R0ZS9YW,5.0
4,B00QO4NAOO,5.0
5,B00PQ1OQ4Y,5.0
6,B00PBIGBOU,5.0
7,B00OGNV5HY,5.0
8,B00OBZNI0O,5.0
9,B00NOD0OTW,5.0


filter by per-user top-k  (1389 users)  
get the top 10 highest rated item per user  
then combine all the top 10 per user lists  
then remove duplicates  
items went from 927 to 546

#### so many trade offs and decisions
how long should the description be?
what should it included?
how detailed? ect.

In [ ]:
# filter by item ids in test set
metadata_df = metadata_df[metadata_df['item_id'].isin(test_df['item_id'])]
metadata_df.head()
print(len(metadata_df), len(test_df["item_id"].unique()))

927 927


In [ ]:
# 3) Content-Based Base (LLM Metadata) - Gemma description generation on SVD candidates only
from concurrent.futures import ThreadPoolExecutor, as_completed

if "candidate_item_ids_svd" not in globals():
    raise RuntimeError("Run the per-user top-k candidate cell before this cell.")

metadata_llm_df = metadata_df.copy()
metadata_llm_df["item_id"] = metadata_llm_df["item_id"].astype(str)
metadata_llm_df["title"] = metadata_llm_df["title"].fillna("").astype(str).str.strip()
metadata_llm_df = metadata_llm_df[metadata_llm_df["title"] != ""].copy()

# Only keep item metadata for CF candidate items
metadata_llm_df = metadata_llm_df[metadata_llm_df["item_id"].isin(candidate_item_ids_svd)].copy()

items_df = (metadata_llm_df[["item_id", "title"]].drop_duplicates(subset=["item_id"]).reset_index(drop=True))

cache_path = Path("csv_files/llm_item_descriptions.csv")
model_name = "gemma3"
deterministic_options = {"temperature": 0.0,"top_p": 1.0,"seed": 42,}

# Keep worker count small to avoid overloading local Ollama
use_parallel = True
max_workers = 3

def generate_short_description(title_text: str) -> str:
    prompt = (
        "Write 1-2 concise sentences describing this video game title for recommender metadata. "
        "Focus on likely genre, gameplay style, and intended audience. "
        "Do not use bullet points. Title: " + title_text
    )
    response = ollama.chat(
        model=model_name,
        messages=[
            {"role": "system", "content": "You generate concise, neutral recommendation metadata."},
            {"role": "user", "content": prompt},
        ],
        options=deterministic_options,
    )
    return response["message"]["content"].strip()

def generate_one_row(row):
    item_id = str(row.item_id)
    title_text = row.title
    try:
        desc = generate_short_description(title_text)
    except Exception as exc:
        desc = ""
        print(f"[WARN] generation failed for item_id={item_id}: {exc}")

    return {
        "item_id": item_id,
        "title": title_text,
        "llm_description": desc,
    }

if cache_path.exists():
    cached_df = pd.read_csv(cache_path, dtype={"item_id": str})
    cached_df = cached_df[["item_id", "title", "llm_description"]].drop_duplicates(subset=["item_id"])
else:
    cached_df = pd.DataFrame(columns=["item_id", "title", "llm_description"])

items_df["item_id"] = items_df["item_id"].astype(str)
done_item_ids = set(cached_df["item_id"].astype(str))
todo_df = items_df[~items_df["item_id"].isin(done_item_ids)].copy()

print(f"Candidate items from SVD top-k: {len(items_df)}")
print(f"Already cached: {len(cached_df)}")
print(f"To generate now: {len(todo_df)}")

new_rows = []
todo_rows = list(todo_df.itertuples(index=False))

if use_parallel and len(todo_rows) > 0:
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(generate_one_row, row) for row in todo_rows]
        for i, future in enumerate(as_completed(futures), start=1):
            new_rows.append(future.result())
            if i % 25 == 0 or i == len(todo_rows):
                print(f"Generated {i}/{len(todo_rows)} new descriptions...")
else:
    for i, row in enumerate(todo_rows, start=1):
        new_rows.append(generate_one_row(row))
        if i % 25 == 0 or i == len(todo_rows):
            print(f"Generated {i}/{len(todo_rows)} new descriptions...")

if new_rows:
    new_df = pd.DataFrame(new_rows)
    final_df = pd.concat([cached_df, new_df], ignore_index=True)
else:
    final_df = cached_df.copy()

final_df = final_df.drop_duplicates(subset=["item_id"], keep="last")
final_df = final_df.sort_values("item_id").reset_index(drop=True)
final_df.to_csv(cache_path, index=False)

llm_descriptions_df = final_df.copy()
print(f"Saved cache to: {cache_path}")
print(f"Cached descriptions total: {len(llm_descriptions_df)}")
llm_descriptions_df.head()

Candidate items from SVD top-k: 543
Already cached: 0
To generate now: 543
Generated 25/543 new descriptions...
Generated 50/543 new descriptions...
Generated 75/543 new descriptions...
Generated 100/543 new descriptions...
Generated 125/543 new descriptions...
Generated 150/543 new descriptions...
Generated 175/543 new descriptions...
Generated 200/543 new descriptions...
Generated 225/543 new descriptions...
Generated 250/543 new descriptions...
Generated 275/543 new descriptions...
Generated 300/543 new descriptions...
Generated 325/543 new descriptions...
Generated 350/543 new descriptions...
Generated 375/543 new descriptions...
Generated 400/543 new descriptions...
Generated 425/543 new descriptions...
Generated 450/543 new descriptions...
Generated 475/543 new descriptions...
Generated 500/543 new descriptions...
Generated 525/543 new descriptions...
Generated 543/543 new descriptions...
Saved cache to: csv_files\llm_item_descriptions.csv
Cached descriptions total: 543


,item_id,title,llm_description
0,B00000JRSB,Final Fantasy VII - PlayStation,Final Fantasy VII – PlayStation is a classic r...
1,B00001X50M,Metal Gear Solid,Metal Gear Solid is a stealth-action game wher...
2,B00001XDUB,Silent Hill,Silent Hill is a psychological horror game tha...
3,B0000296O5,Final Fantasy VIII,Final Fantasy VIII is a classic JRPG offering ...
4,B00004Y57G,Final Fantasy IX,Final Fantasy IX is a classic JRPG offering a ...


#### bro never thinks about computational costs for calling a local LLM 
you don't think about the implication of that many rows
just do a top k bruh